<a href="https://colab.research.google.com/github/NihalChaturvedi7525/Chowmein-Factory-AI/blob/main/Copy_of_AI_Customer_Support_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q langgraph langchain-groq

from langchain_groq import ChatGroq
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


class CustomerState(TypedDict):
    user_input: str
    category: str
    response: str


def classify_query(state):
    message = state["user_input"].lower()

    if "order" in message or "delivery" in message:
        category = "Order"

    elif "refund" in message or "money back" in message:
        category = "Refund"

    elif "complaint" in message or "angry" in message or "damaged" in message:
        category = "Complaint"

    else:
        category = "General"

    return {"category": category}


def order_support(state):
    prompt = f"""
You are a helpful customer support agent.

Customer message:
{state["user_input"]}

Give a short and polite response about the order.
"""
    result = llm.invoke(prompt)
    return {"response": result.content}


def refund_support(state):
    prompt = f"""
You are a customer support agent.

Customer message:
{state["user_input"]}

Explain the refund process simply and politely.
"""
    result = llm.invoke(prompt)
    return {"response": result.content}


def complaint_support(state):
    prompt = f"""
You are a professional customer support agent.

Customer complaint:
{state["user_input"]}

Apologize politely and provide a helpful response.
"""
    result = llm.invoke(prompt)
    return {"response": result.content}


def general_support(state):
    prompt = f"""
You are a helpful customer support agent.

Customer message:
{state["user_input"]}

Answer politely in simple language.
"""
    result = llm.invoke(prompt)
    return {"response": result.content}


def route_query(state):
    if state["category"] == "Order":
        return "order"

    elif state["category"] == "Refund":
        return "refund"

    elif state["category"] == "Complaint":
        return "complaint"

    else:
        return "general"


graph = StateGraph(CustomerState)

graph.add_node("classify", classify_query)
graph.add_node("order", order_support)
graph.add_node("refund", refund_support)
graph.add_node("complaint", complaint_support)
graph.add_node("general", general_support)

graph.add_edge(START, "classify")

graph.add_conditional_edges(
    "classify",
    route_query,
    {
        "order": "order",
        "refund": "refund",
        "complaint": "complaint",
        "general": "general"
    }
)

graph.add_edge("order", END)
graph.add_edge("refund", END)
graph.add_edge("complaint", END)
graph.add_edge("general", END)


app = graph.compile()

print("Customer Support Agent Ready!")


result = app.invoke({
    "user_input": "My order is late",
    "category": "",
    "response": ""
})

print("Category:", result["category"])
print("Response:", result["response"])

Customer Support Agent Ready!
Category: Order
Response: I’m sorry your order hasn’t arrived yet. Let me look into its status right away and get you an update.


In [ ]:

def order_support(state):
    order_status = "Your order is currently out for delivery."

    prompt = f"""
You are a helpful customer support agent.

Customer message:
{state["user_input"]}

Order status:
{order_status}

Give a short and polite response to the customer.
"""

    result = llm.invoke(prompt)

    return {
        "response": result.content
    }

In [ ]:

result = app.invoke({
    "user_input": "Where is my order?",
    "category": "",
    "response": ""
})

print("Category:", result["category"])
print("Response:", result["response"])

Category: Order
Response: Hi there! I’m happy to help. Could you please share your order number or the email you used to place the order? Once I have that, I’ll check the status and let you know where it is. Thank you!


In [ ]:

def complaint_support(state):
    prompt = f"""
You are a professional customer support agent.

Customer complaint:
{state["user_input"]}

Apologize politely and provide a helpful response.
"""

    result = llm.invoke(prompt)

    return {
        "response": result.content +
        "\n\nYour issue has been escalated to a human support agent."
    }

In [ ]:

result = app.invoke({
    "user_input": "My product is damaged and I am angry",
    "category": "",
    "response": ""
})

print("Category:", result["category"])
print("Response:", result["response"])

Category: Complaint
Response: Subject: We’re sorry to hear about your experience – let’s fix this

Dear [Customer Name],

I’m truly sorry to hear that your product arrived damaged. I understand how frustrating this must be, and I appreciate you bringing it to our attention.

**Here’s what we’ll do right away:**

1. **Send a replacement** – We’ll ship a brand‑new unit to you at no additional cost.  
2. **Offer a refund** – If you’d prefer a full refund instead, we can process that immediately.  
3. **Provide a prepaid return label** – We’ll cover the cost of returning the damaged item, so you won’t have to worry about shipping.

**Next steps for you:**

- Please reply to this email with a brief photo of the damage (if possible).  
- Let us know whether you’d like a replacement or a refund.  
- If you choose a replacement, we’ll ship it within 24 hours and send you a tracking number.

We’re committed to making this right and ensuring you’re satisfied with your purchase. Thank you for you

In [ ]:

graph = StateGraph(CustomerState)

graph.add_node("classify", classify_query)
graph.add_node("order", order_support)
graph.add_node("refund", refund_support)
graph.add_node("complaint", complaint_support)
graph.add_node("general", general_support)

graph.add_edge(START, "classify")

graph.add_conditional_edges(
    "classify",
    route_query,
    {
        "order": "order",
        "refund": "refund",
        "complaint": "complaint",
        "general": "general"
    }
)

graph.add_edge("order", END)
graph.add_edge("refund", END)
graph.add_edge("complaint", END)
graph.add_edge("general", END)

app = graph.compile()

print("Updated Customer Support Agent Ready!")

Updated Customer Support Agent Ready!


In [ ]:

while True:
    user_message = input("You: ")

    if user_message.lower() == "exit":
        print("Chat ended.")
        break

    result = app.invoke({
        "user_input": user_message,
        "category": "",
        "response": ""
    })

    print("AI:", result["response"])

You: Hii
AI: Hello! 👋 How can I help you today?
You: I can oder something
AI: Sure! I’d be happy to help you place an order.  
Could you let me know what you’d like to order? If you have a specific item or a list of items, just tell me the names and quantities, and I’ll take care of the rest.  

If you need help finding something or have any questions, feel free to ask!
You: Burger
AI: Sure! How can I help you with burgers today?
You: I want some Maggie
AI: Sure! Which flavor of Maggie would you like? And where would you like it delivered? Let me know and I’ll help you place the order.
You: Ok thank you
AI: You’re welcome! If you have any more questions or need further help, just let me know. Have a great day!
You: Exit
Chat ended.


In [ ]:

tests = [
    "My order is late",
    "I want a refund",
    "My product is damaged and I am angry",
    "How can I contact customer support?"
]

for message in tests:
    result = app.invoke({
        "user_input": message,
        "category": "",
        "response": ""
    })

    print("User:", message)
    print("Category:", result["category"])
    print("AI:", result["response"])
    print("-" * 50)

User: My order is late
Category: Order
AI: I’m sorry for the delay. Your order is currently out for delivery and should arrive shortly. Thank you for your patience!
--------------------------------------------------
User: I want a refund
Category: Refund
AI: Sure thing! Here’s a quick, friendly rundown of how to get a refund:

1. **Reach out to us**  
   • Send an email to support@example.com or use the “Contact Us” form on our website.  
   • Include your order number, the date of purchase, and a brief reason for the refund.

2. **We’ll review your request**  
   • Our team checks the order details and confirms the refund eligibility (most items are refundable within 30 days of delivery, unless they’re marked as final‑sale).

3. **Refund approval**  
   • Once approved, we’ll send you a confirmation email.  
   • The refund will be processed to the original payment method.

4. **Processing time**  
   • It usually takes 3–5 business days for the amount to appear on your statement, dep